<a href="https://colab.research.google.com/github/rsher60/LLM_Codebase/blob/main/Rsher60_Text2SQL_Finetuning_Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers datasets bitsandbytes trl peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 125.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.9/511.9 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Model Loading Best Practices:

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name="meta-llama/Llama-3.2-1B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,  # Preferred over float16
    device_map="auto",
    trust_remote_code=True,
    #attn_implementation="flash_attention_2"  # If available
)


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb):

In [ ]:

print("=== Step 1: Check Base Model Status ===")
print(f"Base model type: {type(model)}")
print(f"Base model training mode: {model.training}")

# Check if base model has any trainable parameters (it shouldn't for LoRA)
base_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
base_total = sum(p.numel() for p in model.parameters())
print(f"Base model - Total params: {base_total:,}, Trainable: {base_trainable:,}")

# Check model device
device = next(model.parameters()).device
print(f"Model device: {device}")

=== Step 1: Check Base Model Status ===
Base model type: <class 'transformers.models.llama.modeling_llama.LlamaForCausalLM'>
Base model training mode: False
Base model - Total params: 1,235,814,400, Trainable: 1,235,814,400
Model device: cuda:0


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left" , trust_remote_code=True)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [ ]:
from datasets import load_dataset
#dataset = load_dataset("MattCoddity/dockerNLcommands")

dataset = load_dataset("gretelai/synthetic_text_to_sql")

dataset


README.md: 0.00B [00:00, ?B/s]

(…)nthetic_text_to_sql_train.snappy.parquet:   0%|          | 0.00/32.4M [00:00<?, ?B/s]

(…)ynthetic_text_to_sql_test.snappy.parquet:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5851 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 100000
    })
    test: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 5851
    })
})

In [ ]:
import pandas as pd

splits = {'train': 'synthetic_text_to_sql_train.snappy.parquet', 'test': 'synthetic_text_to_sql_test.snappy.parquet'}
df = pd.read_parquet("hf://datasets/gretelai/synthetic_text_to_sql/" + splits["train"])

In [ ]:
for i in df['sql_complexity'].unique():
  print(i ,df[df['sql_complexity']==i].shape)

single join (14932, 11)
aggregation (22015, 11)
basic SQL (48466, 11)
window functions (3596, 11)
subqueries (6719, 11)
multiple_joins (2949, 11)
set operations (1050, 11)
CTEs (273, 11)


In [ ]:
stacked_df = pd.DataFrame()

#create a dataframe that at least has 3500 rows

for i in ['single join' ,'aggregation' , 'basic SQL' , 'window functions' , 'subqueries']:

  temp_df = df[df['sql_complexity']==i].sample(2500)
  stacked_df = pd.concat([stacked_df,temp_df ], axis=0)

In [ ]:
stacked_df['instruction'] = '''You are an expert SQL query generator. Your task is to translate natural language questions into accurate, efficient, and syntactically correct SQL queries based on the provided database schema and sample data.

## Core Responsibilities:
1. **Analyze the natural language question** to understand the exact data requirements
2. **Use the provided table schema and sample data** to construct precise queries
3. **Generate clean, executable SQL** without explanations or commentary

## Database Context:
You will receive:
- **Table schemas** with column names, data types, and relationships
- **Sample rows** showing actual data patterns and formats
- **Natural language question** specifying the data requirement

## Query Construction Guidelines:

### Column Selection:
- Select **specific columns** that answer the question
- Use `SELECT *` only when explicitly requested or when all columns are needed
- Reference actual column names from the provided schema

### Data Filtering:
- Apply appropriate **WHERE clauses** based on the question
- Use correct data types and formats as shown in sample data
- Handle date formats, text patterns, and numeric ranges accurately

### Aggregations & Grouping:
- Use `COUNT()`, `SUM()`, `AVG()`, `MAX()`, `MIN()` when the question implies aggregation
- Apply `GROUP BY` for categorical breakdowns
- Include `ORDER BY` for sorting requirements

### Table Relationships:
- Identify and use proper **JOIN** operations when questions span multiple tables
- Reference foreign key relationships from the schema
- Use appropriate join types (INNER, LEFT, RIGHT) based on data requirements

### Query Optimization:
- Write efficient queries that minimize unnecessary operations
- Use appropriate **LIMIT** clauses for "top N" requests
- Apply indexes-friendly conditions when possible

## Output Format:
- Provide **only the SQL query** - no explanations, comments, or additional text
- Ensure the query is **syntactically correct** and executable
- Use proper SQL formatting and indentation

## Example Patterns:

**Counting Records:**
```sql
SELECT COUNT(*) FROM table_name WHERE condition;
```

**Top N Results:**
```sql
SELECT column1, column2 FROM table_name ORDER BY column DESC LIMIT N;
```

**Aggregation by Category:**
```sql
SELECT category, SUM(amount) FROM table_name GROUP BY category;
```

**Multi-table Joins:**
```sql
SELECT t1.col1, t2.col2 FROM table1 t1 JOIN table2 t2 ON t1.id = t2.foreign_id WHERE condition;
```

**Date Range Filtering:**
```sql
SELECT * FROM table_name WHERE date_column BETWEEN 'start_date' AND 'end_date';
```

---

'''

In [ ]:
stacked_df.head()

,id,domain,domain_description,sql_complexity,sql_complexity_description,sql_task_type,sql_task_type_description,sql_prompt,sql_context,sql,sql_explanation,instruction
62803,89824,telecommunications,"Mobile and broadband subscriber data, network ...",single join,"only one join (specify inner, outer, cross)",analytics and reporting,"generating reports, dashboards, and analytical...",Find subscribers who have both mobile and broa...,CREATE TABLE mobile_subscribers (subscriber_id...,SELECT m.subscriber_id FROM mobile_subscribers...,This query joins the mobile_subscribers and br...,You are an expert SQL query generator. Your ta...
51890,19329,fitness industry,"Workout data, membership demographics, wearabl...",single join,"only one join (specify inner, outer, cross)",analytics and reporting,"generating reports, dashboards, and analytical...",What is the maximum duration of a workout for ...,"CREATE TABLE Workouts (WorkoutID INT, Duration...","SELECT Members.Name, MAX(Workouts.Duration) FR...",This query calculates the maximum duration of ...,You are an expert SQL query generator. Your ta...
18440,10450,agriculture,"Comprehensive data on agroecology, food justic...",single join,"only one join (specify inner, outer, cross)",analytics and reporting,"generating reports, dashboards, and analytical...",List all urban gardens in the 'food_justice' s...,CREATE SCHEMA if not exists food_justice; use ...,"SELECT urban_gardens.name, community_orgs.name...",This query performs an inner join on the urban...,You are an expert SQL query generator. Your ta...
27639,43817,gaming industry,"Player analytics, game performance metrics, eS...",single join,"only one join (specify inner, outer, cross)",analytics and reporting,"generating reports, dashboards, and analytical...",What is the minimum player score for each game?,"CREATE TABLE MinPlayerScores (player_id INT, g...","SELECT G.game_name, MIN(MPS.player_score) as m...",The SQL query first joins the MinPlayerScores ...,You are an expert SQL query generator. Your ta...
59328,57000,healthcare,"Healthcare data on mental health parity, cultu...",single join,"only one join (specify inner, outer, cross)",analytics and reporting,"generating reports, dashboards, and analytical...",Show the list of patients who are of Asian or ...,"CREATE TABLE patient (id INT PRIMARY KEY, name...","SELECT p.id, p.name FROM patient p JOIN mental...",This SQL query joins 'patient' and 'mental_hea...,You are an expert SQL query generator. Your ta...


In [ ]:
from datasets import Dataset, DatasetDict
Dataset.from_pandas(stacked_df)

Dataset({
    features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'instruction', '__index_level_0__'],
    num_rows: 12500
})

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
import numpy as np
def convert_df_to_datasetdict(df: pd.DataFrame, split_name: str = "train") -> DatasetDict:
    """
    Converts a Pandas DataFrame into a Hugging Face DatasetDict.

    Args:
        df (pd.DataFrame): The input Pandas DataFrame.
        split_name (str): The name of the split to assign to the dataset (e.g., "train", "validation", "test").
                          Defaults to "train".

    Returns:
        DatasetDict: A Hugging Face DatasetDict containing the DataFrame as a Dataset.
    """
    if not isinstance(df, pd.DataFrame):
        print("Error: Input 'df' must be a Pandas DataFrame.")
        return DatasetDict()

    # Step 1: Convert the Pandas DataFrame to a Hugging Face Dataset
    # The Dataset.from_pandas() method is efficient for this conversion.
    # It automatically infers the features (columns and their types).
    hf_dataset = Dataset.from_pandas(df)

    # Step 2: Create a DatasetDict from the Dataset
    # A DatasetDict is a dictionary-like object that can hold multiple Dataset objects,
    # typically representing different splits (train, validation, test).
    # Here, we're putting our single DataFrame-derived Dataset into a 'train' split.
    dataset_dict = DatasetDict({split_name: hf_dataset})

    print(f"Successfully converted DataFrame to DatasetDict with split '{split_name}'.")
    return dataset_dict


dataset = convert_df_to_datasetdict(stacked_df, "train")

Successfully converted DataFrame to DatasetDict with split 'train'.


In [ ]:
from datasets import DatasetDict
from sklearn.model_selection import train_test_split

train_val_split = dataset['train'].train_test_split(test_size=0.2, seed=42)

dataset = DatasetDict(
    {'train':train_val_split['train'],
     'validation': train_val_split['test']}
)

dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'instruction', '__index_level_0__'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'instruction', '__index_level_0__'],
        num_rows: 2500
    })
})

In [ ]:
def to_chat_template(example):

  #sql_prompt is the user input, sql_context is the table definition, and the SQL is the actual query.

  messages =[
      {"role":'system', 'content':example['instruction']},
      {"role":'user', 'content':'**Schema and Sample Data:**' + example['sql_context'] + '\n' + '**Natural Language Question:**' + example['sql_prompt']  },
      {"role":'assistant', 'content': '**SQL:**' +  example['sql']}
  ]

  return {'text': messages}

dataset = dataset.map(to_chat_template)

dataset

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'instruction', '__index_level_0__', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'instruction', '__index_level_0__', 'text'],
        num_rows: 2500
    })
})

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
tokenizer.apply_chat_template(dataset['train']['text'][0], tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 08 Aug 2025\n\nYou are an expert SQL query generator. Your task is to translate natural language questions into accurate, efficient, and syntactically correct SQL queries based on the provided database schema and sample data.\n\n## Core Responsibilities:\n1. **Analyze the natural language question** to understand the exact data requirements\n2. **Use the provided table schema and sample data** to construct precise queries\n3. **Generate clean, executable SQL** without explanations or commentary\n\n## Database Context:\nYou will receive:\n- **Table schemas** with column names, data types, and relationships\n- **Sample rows** showing actual data patterns and formats\n- **Natural language question** specifying the data requirement\n\n## Query Construction Guidelines:\n\n### Column Selection:\n- Select **specific columns** that answer the question\n- Use `SELECT *` only when e

In [ ]:
def apply_chat_temp(example):

  new_text = tokenizer.apply_chat_template(example['text'], tokenize=False)
  return {'text': new_text}

dataset = dataset.map(apply_chat_temp)
dataset

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'instruction', '__index_level_0__', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation', 'instruction', '__index_level_0__', 'text'],
        num_rows: 2500
    })
})

In [ ]:
dataset['train']['text'][1]


# I have added the cutting knowledge date to the text

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 08 Aug 2025\n\nYou are an expert SQL query generator. Your task is to translate natural language questions into accurate, efficient, and syntactically correct SQL queries based on the provided database schema and sample data.\n\n## Core Responsibilities:\n1. **Analyze the natural language question** to understand the exact data requirements\n2. **Use the provided table schema and sample data** to construct precise queries\n3. **Generate clean, executable SQL** without explanations or commentary\n\n## Database Context:\nYou will receive:\n- **Table schemas** with column names, data types, and relationships\n- **Sample rows** showing actual data patterns and formats\n- **Natural language question** specifying the data requirement\n\n## Query Construction Guidelines:\n\n### Column Selection:\n- Select **specific columns** that answer the question\n- Use `SELECT *` only when e

In [ ]:
# Correct tokenization function for fine-tuning
def tokenize_fn(example):
    tokenized = tokenizer(
        example['text'],
        truncation=True,        # Essential: handle long sequences
        max_length=512,         # Set appropriate max length
        padding=False,          # Let DataCollator handle padding dynamically
        return_tensors=None     # Return lists, not tensors
    )

    # CRITICAL: Add labels for supervised learning
    # For causal language modeling, labels = input_ids
    tokenized['labels'] = tokenized['input_ids'].copy()

    return tokenized

# Apply tokenization and remove ALL original columns (including 'text')
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=['id', 'domain', 'domain_description', 'sql_complexity',
                   'sql_complexity_description', 'sql_task_type',
                   'sql_task_type_description', 'sql_prompt', 'sql_context',
                   'sql', 'sql_explanation', 'instruction', '__index_level_0__',
                   'text']  # Make sure 'text' is included here
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [ ]:
from transformers import DataCollatorForLanguageModeling


data_collator = DataCollatorForLanguageModeling(tokenizer, #use the Llama tokenizer
                                                mlm=False,  #MLm means masked language modelling, it should be False for Causal Modelling -  Causal Language Modeling (like GPT/Llama) and Masked Language Modelling is like BERT
                                                return_tensors="pt") # REturn the pytorch tensors



In [ ]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|eot_id|>'

In [ ]:
from torch.utils.data import DataLoader

data_loader = DataLoader(
    tokenized_dataset['train'],
    batch_size = 2,
    collate_fn = data_collator
)

for step, batch in enumerate(data_loader):
  print(f"Batch {step}")
  print("input_ids.shape:" , batch["input_ids"].shape)
  print("attention_mask.shape:", batch["attention_mask"].shape)


  if step >= 3:
    break

Batch 0
input_ids.shape: torch.Size([2, 512])
attention_mask.shape: torch.Size([2, 512])
Batch 1
input_ids.shape: torch.Size([2, 512])
attention_mask.shape: torch.Size([2, 512])
Batch 2
input_ids.shape: torch.Size([2, 512])
attention_mask.shape: torch.Size([2, 512])
Batch 3
input_ids.shape: torch.Size([2, 512])
attention_mask.shape: torch.Size([2, 512])


In [ ]:
from peft import LoraConfig, get_peft_model

import copy

#model_8bit_clone = copy.deepcopy(model_8bit) - Claude asked me to remove it


from peft import LoraConfig, get_peft_model

# DON'T use deepcopy - use the original model directly
lora_config = LoraConfig(
    r=64,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Create LoRA directly from model_8bit (no copying)
model_8bit_lora = get_peft_model(model, lora_config)
model_8bit_lora.print_trainable_parameters()

/usr/local/lib/python3.11/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 45,088,768 || all params: 1,280,903,168 || trainable%: 3.5201


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_steps=10,
    eval_strategy="steps",
    save_steps=20,
    save_strategy="steps",

    # Don't set num_train_epochs at all
    max_steps=60,

    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=5,
    fp16=False,
    bf16=True,
    gradient_accumulation_steps=4,
    optim="adamw_torch",
    logging_steps=10,
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

# TRAINER SETUP (unchanged - this looks correct)
trainer = SFTTrainer(
    model=model_8bit_lora,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    args=training_args,  # or training_args_alternative
    data_collator=data_collator,
    processing_class=tokenizer,  # Correct for new TRL version
)

# VERIFICATION: Check what the trainer will actually do
print("=== TRAINING CONFIGURATION VERIFICATION ===")
print(f"Max steps setting: {training_args.max_steps}")
print(f"Num epochs setting: {training_args.num_train_epochs}")
print(f"Dataset size: {len(tokenized_dataset['train'])}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Expected total steps: {training_args.max_steps}")

# Calculate what steps SHOULD be with your dataset
effective_batch_size = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = len(tokenized_dataset['train']) // effective_batch_size
print(f"Steps per epoch would be: {steps_per_epoch}")
print(f"But we're limiting to: {training_args.max_steps} steps")

# FINAL CHECK: Print trainer's planned steps
print(f"\nTrainer will run for: {trainer.args.max_steps} steps")
if trainer.args.max_steps == 60:
    print("✅ Configuration fixed - will train for 60 steps")
else:
    print("❌ Still showing wrong step count - try alternative config")

# START TRAINING
print("\n=== STARTING TRAINING ===")
trainer.train()

Truncating train dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/2500 [00:00<?, ? examples/s]

=== TRAINING CONFIGURATION VERIFICATION ===
Max steps setting: 60
Num epochs setting: 3.0
Dataset size: 10000
Effective batch size: 8
Expected total steps: 60
Steps per epoch would be: 1250
But we're limiting to: 60 steps

Trainer will run for: 60 steps
✅ Configuration fixed - will train for 60 steps

=== STARTING TRAINING ===


Step,Training Loss,Validation Loss
10,2.050800,1.838434
20,1.617700,1.376572
30,1.193800,0.988966
40,0.835200,0.669258
50,0.561400,0.454101
60,0.408000,0.378128


TrainOutput(global_step=60, training_loss=1.1111385345458984, metrics={'train_runtime': 474.9245, 'train_samples_per_second': 1.011, 'train_steps_per_second': 0.126, 'total_flos': 1501448424652800.0, 'train_loss': 1.1111385345458984})

In [ ]:
# STEP 3: POST-TRAINING EVALUATION

from transformers import pipeline
print("\n=== POST-TRAINING EVALUATION ===")

# Evaluate final model
eval_results = trainer.evaluate()
print("Final evaluation results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")

# STEP 4: SAVE THE FINE-TUNED MODEL
print("\n=== SAVING MODEL ===")

# Save LoRA adapter
lora_output_dir = "./lora_adapters"
model_8bit_lora.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print(f"✅ LoRA adapter saved to {lora_output_dir}")

# STEP 5: TEST THE FINE-TUNED MODEL
print("\n=== TESTING FINE-TUNED MODEL ===")

# Method 1: Test with LoRA adapter (memory efficient)
from transformers import pipeline
import torch

pipe = pipeline(
        "text-generation",
        model=model_8bit_lora,
        tokenizer=tokenizer,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )


def test_model_with_lora():
    # Create pipeline with LoRA model
    pipe = pipeline(
        "text-generation",
        model=model_8bit_lora,
        tokenizer=tokenizer,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    # Test with a sample from your domain
    test_messages = [
        {
            "role": "system",
            "content": """You are an expert SQL query generator. Your task is to translate natural language questions into accurate and efficient SQL queries."""
        },
        {
            "role": "user",
            "content": "Show me all customers who made purchases in the last 30 days."
        }
    ]

    outputs = pipe(
        test_messages,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True
    )

    print("Sample output:")
    print(outputs[0]["generated_text"][-1]['content'])

# Run the test
test_model_with_lora()

# STEP 6: COMPARE WITH BASE MODEL (Optional but recommended)
print("\n=== BASELINE COMPARISON ===")

def compare_with_baseline():
    # Load base model for comparison
    from transformers import AutoModelForCausalLM

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    base_pipe = pipeline(
        "text-generation",
        model=base_model,
        tokenizer=tokenizer,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    test_prompt = "Convert this to SQL: Show me all customers who made purchases in the last 30 days."

    print("Base model output:")
    base_output = base_pipe(test_prompt, max_new_tokens=100, temperature=0.1)
    print(base_output[0]['generated_text'][len(test_prompt):])

    print("\nFine-tuned model output:")
    ft_output = pipe(test_prompt, max_new_tokens=100, temperature=0.1)
    print(ft_output[0]['generated_text'][len(test_prompt):])

# Uncomment to run comparison
compare_with_baseline()

# STEP 7: CREATE MERGED MODEL (For deployment)
print("\n=== CREATING MERGED MODEL FOR DEPLOYMENT ===")

# Option A: Merge and save locally
def create_merged_model():
    from transformers import AutoModelForCausalLM
    from peft import PeftModel

    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # Load with LoRA adapter
    merged_model = PeftModel.from_pretrained(base_model, lora_output_dir)

    # Merge LoRA weights into base model
    merged_model = merged_model.merge_and_unload()

    # Save merged model
    merged_output_dir = "./merged_model"
    merged_model.save_pretrained(merged_output_dir)
    tokenizer.save_pretrained(merged_output_dir)

    print(f"✅ Merged model saved to {merged_output_dir}")
    return merged_model

# Create merged model
merged_model = create_merged_model()

# STEP 8: PUSH TO HUB (Optional)
print("\n=== PUSHING TO HUGGING FACE HUB ===")

def push_to_hub(hub_model_name):
    """
    Push your fine-tuned model to Hugging Face Hub
    Replace 'your_username' with your actual username
    """
    try:
        # Push merged model
        merged_model.push_to_hub(hub_model_name)
        tokenizer.push_to_hub(hub_model_name)
        print(f"✅ Model pushed to https://huggingface.co/{hub_model_name}")

        # Also push LoRA adapter separately
        lora_hub_name = f"{hub_model_name}-lora"
        model_8bit_lora.push_to_hub(lora_hub_name)
        print(f"✅ LoRA adapter pushed to https://huggingface.co/{lora_hub_name}")

    except Exception as e:
        print(f"❌ Error pushing to hub: {e}")
        print("Make sure you're logged in: huggingface-cli login")

# PUSH TO HUB - UNCOMMENT AND CUSTOMIZE THE MODEL NAME
hub_model_name = "rsher60/llama3.2-1B-text2sql-finetuned"  # Change this to your username/model-name

# Login check and push
print("Checking Hugging Face login status...")
try:
    from huggingface_hub import whoami
    user_info = whoami()
    print(f"✅ Logged in as: {user_info['name']}")

    # Push the models
    push_to_hub(hub_model_name)

except Exception as e:
    print(f"❌ Not logged in to Hugging Face Hub: {e}")
    print("To login, run in terminal: huggingface-cli login")
    print("Then get your token from: https://huggingface.co/settings/tokens")
    print("After logging in, you can push with:")
    print(f"push_to_hub('{hub_model_name}')")

# STEP 9: PERFORMANCE EVALUATION
print("\n=== COMPREHENSIVE EVALUATION ===")

def evaluate_text2sql_performance():
    """
    Evaluate on your specific text2SQL test cases
    """
    # Sample test cases - replace with your actual test data
    test_cases = [
        {
            "question": "How many customers are there?",
            "expected_sql": "SELECT COUNT(*) FROM customers;",
            "context": "Table: customers (id, name, email, signup_date)"
        },
        {
            "question": "Show me the top 5 most expensive products",
            "expected_sql": "SELECT * FROM products ORDER BY price DESC LIMIT 5;",
            "context": "Table: products (id, name, price, category)"
        }
    ]

    correct_predictions = 0
    total_predictions = len(test_cases)

    for i, test_case in enumerate(test_cases):
        prompt = f"""Convert this to SQL: {test_case['question']}
Context: {test_case['context']}
SQL:"""

        # Generate SQL
        outputs = pipe(
            prompt,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True
        )

        predicted_sql = outputs[0]['generated_text'][len(prompt):].strip()

        print(f"\nTest Case {i+1}:")
        print(f"Question: {test_case['question']}")
        print(f"Expected: {test_case['expected_sql']}")
        print(f"Predicted: {predicted_sql}")

        # Simple evaluation (you might want more sophisticated matching)
        if test_case['expected_sql'].lower() in predicted_sql.lower():
            correct_predictions += 1
            print("✅ Match")
        else:
            print("❌ No match")

    accuracy = correct_predictions / total_predictions
    print(f"\nOverall Accuracy: {accuracy:.2%} ({correct_predictions}/{total_predictions})")

# Run evaluation
evaluate_text2sql_performance()

# STEP 10: CLEANUP AND MONITORING SETUP
print("\n=== CLEANUP AND NEXT STEPS ===")

print("Training completed! Next steps:")
print("1. ✅ Model training finished")
print("2. ✅ Model saved and tested")
print("3. ✅ Performance evaluation completed")
print("4. 📝 Document your results and hyperparameters")
print("5. 🚀 Deploy model using your preferred serving framework")
print("6. 📊 Set up monitoring for production use")
print("7. 🔄 Plan for iterative improvements")

# Memory cleanup
import gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Memory cleaned up")

print("\n🎉 Fine-tuning workflow completed successfully!")


=== POST-TRAINING EVALUATION ===


Step,Training Loss,Validation Loss
10,0.265100,0.115133


Final evaluation results:
eval_loss: 0.11513267457485199

=== SAVING MODEL ===


Device set to use cuda:0
Device set to use cuda:0


✅ LoRA adapter saved to ./lora_adapters

=== TESTING FINE-TUNED MODEL ===
Sample output:
SELECT * FROM customers WHERE created_at BETWEEN NOW() - INTERVAL 30 DAY

=== BASELINE COMPARISON ===


...

✅ Model pushed to https://huggingface.co/rsher60/llama3.2-1B-text2sql-finetuned


Processing Files (0 / 0)                : |          |  0.00B /  0.00B            

New Data Upload                         : |          |  0.00B /  0.00B            

  ...p955s1r2f/adapter_model.safetensors:   0%|          | 23.5kB /  180MB            

✅ LoRA adapter pushed to https://huggingface.co/rsher60/llama3.2-1B-text2sql-finetuned-lora

=== COMPREHENSIVE EVALUATION ===

Test Case 1:
Question: How many customers are there?
Expected: SELECT COUNT(*) FROM customers;
Predicted: SELECT COUNT(*) FROM customers WHERE signup_date > '2020-01-01'

### Explanation

This query counts the number of rows in the `customers` table where the `signup_date` is greater than the specified date ('2020-01-01'). The `COUNT(*)` function returns the total number of rows that meet the condition.

### Example Use Case

Suppose you want to know how many customers signed up in January 2020 or later. You can use the above query
❌ No match

Test Case 2:
Question: Show me the top 5 most expensive products
Expected: SELECT * FROM products ORDER BY price DESC LIMIT 5;
Predicted: SELECT name FROM products ORDER BY price DESC LIMIT 5

Here's the code:
```sql
SELECT name FROM products ORDER BY price DESC LIMIT 5
```
Explanation:

* `SELECT name`: Selects the `name

In [ ]:
# STEP 1: PRE-TRAINING VALIDATION (Run this BEFORE trainer.train())
print("=== PRE-TRAINING VALIDATION ===")

# Fix the typo in output_dir first
training_args.output_dir = "./results"  # Fix "resutls" typo

# Verify trainable parameters
print("Trainable parameters check:")
model_8bit_lora.print_trainable_parameters()
trainable_params = model_8bit_lora.num_parameters(only_trainable=True)
if trainable_params == 0:
    print("❌ CRITICAL: 0 trainable parameters! Need to fix LoRA setup first.")
    # Don't proceed with training
else:
    print(f"✅ Trainable parameters: {trainable_params:,}")

# Test data loading
print("\nData validation:")
print(f"Train samples: {len(tokenized_dataset['train'])}")
print(f"Validation samples: {len(tokenized_dataset['validation'])}")

# Test a small batch
try:
    sample_batch = tokenized_dataset['train'][:2]
    print("✅ Data loading successful")
except Exception as e:
    print(f"❌ Data loading error: {e}")

# STEP 2: EXECUTE TRAINING
print("\n=== STARTING TRAINING ===")
trainer.train()

# STEP 3: POST-TRAINING EVALUATION
print("\n=== POST-TRAINING EVALUATION ===")

# Evaluate final model
eval_results = trainer.evaluate()
print("Final evaluation results:")
for key, value in eval_results.items():
    print(f"{key}: {value}")

# STEP 4: SAVE THE FINE-TUNED MODEL
print("\n=== SAVING MODEL ===")

# Save LoRA adapter
lora_output_dir = "./lora_adapters"
model_8bit_lora.save_pretrained(lora_output_dir)
tokenizer.save_pretrained(lora_output_dir)
print(f"✅ LoRA adapter saved to {lora_output_dir}")

# STEP 5: TEST THE FINE-TUNED MODEL
print("\n=== TESTING FINE-TUNED MODEL ===")

# Method 1: Test with LoRA adapter (memory efficient)
from transformers import pipeline
import torch

def test_model_with_lora():
    # Create pipeline with LoRA model
    pipe = pipeline(
        "text-generation",
        model=model_8bit_lora,
        tokenizer=tokenizer,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    # Test with a sample from your domain
    test_messages = [
        {
            "role": "system",
            "content": """You are an expert SQL query generator. Your task is to translate natural language questions into accurate and efficient SQL queries."""
        },
        {
            "role": "user",
            "content": "Show me all customers who made purchases in the last 30 days."
        }
    ]

    outputs = pipe(
        test_messages,
        max_new_tokens=256,
        temperature=0.1,
        do_sample=True
    )

    print("Sample output:")
    print(outputs[0]["generated_text"][-1]['content'])

# Run the test
test_model_with_lora()

# STEP 6: COMPARE WITH BASE MODEL (Optional but recommended)
print("\n=== BASELINE COMPARISON ===")

def compare_with_baseline():
    # Load base model for comparison
    from transformers import AutoModelForCausalLM

    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    base_pipe = pipe(
        "text-generation",
        model=base_model,
        tokenizer=tokenizer,
        torch_dtype=torch.bfloat16,
        device_map="auto"
    )

    test_prompt = "Convert this to SQL: Show me all customers who made purchases in the last 30 days."

    print("Base model output:")
    base_output = base_pipe(test_prompt, max_new_tokens=100, temperature=0.1)
    print(base_output[0]['generated_text'][len(test_prompt):])

    print("\nFine-tuned model output:")
    ft_output = pipe(test_prompt, max_new_tokens=100, temperature=0.1)
    print(ft_output[0]['generated_text'][len(test_prompt):])

# Uncomment to run comparison
# compare_with_baseline()

# STEP 7: CREATE MERGED MODEL (For deployment)
print("\n=== CREATING MERGED MODEL FOR DEPLOYMENT ===")

# Option A: Merge and save locally
def create_merged_model():
    from transformers import AutoModelForCausalLM
    from peft import PeftModel

    # Load base model
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )

    # Load with LoRA adapter
    merged_model = PeftModel.from_pretrained(base_model, lora_output_dir)

    # Merge LoRA weights into base model
    merged_model = merged_model.merge_and_unload()

    # Save merged model
    merged_output_dir = "./merged_model"
    merged_model.save_pretrained(merged_output_dir)
    tokenizer.save_pretrained(merged_output_dir)

    print(f"✅ Merged model saved to {merged_output_dir}")
    return merged_model

# Create merged model
merged_model = create_merged_model()

# STEP 8: PUSH TO HUB (Optional)
print("\n=== PUSHING TO HUGGING FACE HUB ===")

def push_to_hub(hub_model_name):
    """
    Push your fine-tuned model to Hugging Face Hub
    Replace 'your_username' with your actual username
    """
    try:
        # Push merged model
        merged_model.push_to_hub(hub_model_name)
        tokenizer.push_to_hub(hub_model_name)
        print(f"✅ Model pushed to https://huggingface.co/{hub_model_name}")

        # Also push LoRA adapter separately
        lora_hub_name = f"{hub_model_name}-lora"
        model_8bit_lora.push_to_hub(lora_hub_name)
        print(f"✅ LoRA adapter pushed to https://huggingface.co/{lora_hub_name}")

    except Exception as e:
        print(f"❌ Error pushing to hub: {e}")
        print("Make sure you're logged in: huggingface-cli login")

# Uncomment and customize to push to hub
# push_to_hub("your_username/llama3.2-text2sql-finetuned")

# STEP 9: PERFORMANCE EVALUATION
print("\n=== COMPREHENSIVE EVALUATION ===")

def evaluate_text2sql_performance():
    """
    Evaluate on your specific text2SQL test cases
    """
    # Sample test cases - replace with your actual test data
    test_cases = [
        {
            "question": "How many customers are there?",
            "expected_sql": "SELECT COUNT(*) FROM customers;",
            "context": "Table: customers (id, name, email, signup_date)"
        },
        {
            "question": "Show me the top 5 most expensive products",
            "expected_sql": "SELECT * FROM products ORDER BY price DESC LIMIT 5;",
            "context": "Table: products (id, name, price, category)"
        }
    ]

    correct_predictions = 0
    total_predictions = len(test_cases)

    for i, test_case in enumerate(test_cases):
        prompt = f"""Convert this to SQL: {test_case['question']}
Context: {test_case['context']}
SQL:"""

        # Generate SQL
        outputs = pipe(
            prompt,
            max_new_tokens=100,
            temperature=0.1,
            do_sample=True
        )

        predicted_sql = outputs[0]['generated_text'][len(prompt):].strip()

        print(f"\nTest Case {i+1}:")
        print(f"Question: {test_case['question']}")
        print(f"Expected: {test_case['expected_sql']}")
        print(f"Predicted: {predicted_sql}")

        # Simple evaluation (you might want more sophisticated matching)
        if test_case['expected_sql'].lower() in predicted_sql.lower():
            correct_predictions += 1
            print("✅ Match")
        else:
            print("❌ No match")

    accuracy = correct_predictions / total_predictions
    print(f"\nOverall Accuracy: {accuracy:.2%} ({correct_predictions}/{total_predictions})")

# Run evaluation
evaluate_text2sql_performance()

# STEP 10: CLEANUP AND MONITORING SETUP
print("\n=== CLEANUP AND NEXT STEPS ===")

print("Training completed! Next steps:")
print("1. ✅ Model training finished")
print("2. ✅ Model saved and tested")
print("3. ✅ Performance evaluation completed")
print("4. 📝 Document your results and hyperparameters")
print("5. 🚀 Deploy model using your preferred serving framework")
print("6. 📊 Set up monitoring for production use")
print("7. 🔄 Plan for iterative improvements")

# Memory cleanup
import gc
gc.collect()
torch.cuda.empty_cache()
print("\n✅ Memory cleaned up")

print("\n🎉 Fine-tuning workflow completed successfully!")

=== PRE-TRAINING VALIDATION ===
Trainable parameters check:
trainable params: 45,088,768 || all params: 1,280,903,168 || trainable%: 3.5201
✅ Trainable parameters: 45,088,768

Data validation:
Train samples: 10000
Validation samples: 2500
✅ Data loading successful

=== STARTING TRAINING ===


Step,Training Loss,Validation Loss


KeyboardInterrupt: 